# Micropolar Piezoelectric Harvester — Topology Optimisation (MMA/NLopt) using Gridap

In [1]:
# FEA
using Gridap
using Gridap.Geometry
using Gridap.Fields
using Gridap.TensorValues
using Gridap.CellData
using Gridap.FESpaces
using GridapTopOpt

# Meshing
using Gmsh
using GridapGmsh

using LinearAlgebra
using SparseArrays

# Gradient (adjoint)
using ChainRulesCore, Zygote
import ChainRulesCore: rrule

# Optimiser — MMA via NLopt
using NLopt
using Printf

In [2]:
const L  = 30#3e-2          # plate length (m)
const H = L/3

h  = 0.1#1e-4                # plate thickness (m)

penal_uu = 3             # stiffness penalisation
penal_uφ = 6             # coupling penalisation  
penal_φφ = 4             # coupling penalisation 
penal_P  = 1             # polarisation penalisation
penal_flexo = 4

w_obj = 0.5
volfrac  = 0.4
F = 1e2
forVec = VectorValue(0.0, -F)

VectorValue{2, Float64}(0.0, -100.0)

In [3]:
const E_mod = 83161.9926199262    # Young's modulus (Pa)
const ν_mod = 0.35885608856088563      # Poisson's ratio
const G     = E_mod / (2*(1 + ν_mod))

const l = L/50        
const N = 0.5         

const λₘₐₜ = E_mod * ν_mod / (1 - ν_mod^2)        # plane stress
const κₘₐₜ = 2*G * N^2 / (1 - N^2)
const μₘₐₜ = G * (1 - 2*N^2) / (1 - N^2)
const γₘₐₜ = 4*G * l^2

function σ_Bmod(ε)
    σM = (λₘₐₜ*tr(ε)*one(ε) + (2*μₘₐₜ + κₘₐₜ)*(ε))
    return σM
end

function σ_piezo(∇)
    σ = ∇ ⋅ e_mat
    return  σ
end 


function D_piezo(∇)
    σ = K_mat ⋅ ∇
    return  σ
end

function D_elas(ε)
    σ = e_mat ⋅² ε
    return  σ
end

E_Matrx = TensorValue(0,1,-1,0)

function ε_Skw(∇,θ)
    ∇ᵀ = transpose(∇)
    w = (0.5*(∇ᵀ - ∇)) - (E_Matrx*θ)
    return w
end

function σ_Cmod(ϵ_skew)
    σM = κₘₐₜ*ϵ_skew
    return σM
end

function M_mod(∇)
    M = γₘₐₜ*∇
    return M
end

function Skw(u,θ)
    ∇ᵀ = transpose(∇(u))
    w = (0.5*(∇ᵀ - ∇(u)) - (E_Matrx*θ))
    return w
end

function Em(p)
    Em = p ^ penal
    return Em
end 

function M_Flexo(∇)
    return (transpose(F_mat) ⋅ ∇)
end

function D_Flexo(∇) 
    return (F_mat ⋅ ∇)
end

D_Flexo (generic function with 1 method)

In [4]:
nx = 225
ny = 75
domain = (0,L,0,H)
model = CartesianDiscreteModel(domain,(nx,ny))
# f_Γ_D1(x) = (x[1] ≈ 0.0) 
# f_Γ_D2(x) = (x[2] ≈ 0.0) 
# f_Γ_D3(x) = (x[2] ≈ H)   
f_Γ_D4(x) = (x[2] ≈ H) && (L/2 - 1*L/nx <= x[1] <= L/2 + 1*L/nx) 
# update_labels!(1,model,f_Γ_D1,"LeftFixedBoundary")
# update_labels!(2,model,f_Γ_D2,"BottomBoundary")
# update_labels!(3,model,f_Γ_D3,"TopBoundary")
update_labels!(4,model,f_Γ_D4,"LoadBoundary")

In [5]:
path = "./MicropolarHArvester_SimplySupported_PZT/penal_flexo_$(penal_flexo)/w_obj$(w_obj)/N_$(N)_l_$(l)/"
mkpath(path)
cd(path)

In [6]:
writevtk(model, "MicropolarActuatorPlate")

3-element Vector{Vector{String}}:
 ["MicropolarActuatorPlate_0.vtu"]
 ["MicropolarActuatorPlate_1.vtu"]
 ["MicropolarActuatorPlate_2.vtu"]

### Piezoelectric constitutive tensors (PZT-4, plane stress) — unchanged from elastic version

In [7]:
const p = VectorValue(1,0)

I = [1 0; 0 1]
K1 = zeros(2,2,2)
K2 = zeros(2,2,2)
K3 = zeros(2,2,2)
C1 = -6.98e-3
C2 = -6.06e-3
C3 = 16.00e-3

const K₁₁_mat = 6e-9
const K₂₂_mat = 5.47e-9
const K_mat = TensorValue(K₁₁_mat,0.0,0.0, K₁₁_mat)

for i = 1:2, j = 1:2, k= 1:2
    K1[i,j,k] = p[i]*I[j,k]
end
for i = 1:2, j = 1:2, k= 1:2
    K2[i,j,k] = p[i]*p[j]*p[k]
end
for i = 1:2, j = 1:2, k= 1:2
    K3[i,j,k] = 0.5*(p[j]*I[i,k] + p[k]*I[i,j])
end
K = C1*K1+C2*K2+C3*K3
vals = zeros(2,2,2);
    vals[1,:,:] .= K[1,:,:]
    vals[2,:,:] .= K[2,:,:]
const e_mat = ThirdOrderTensorValue(vals ...)

ThirdOrderTensorValue{2, 2, 2, Float64, 8}(0.002960000000000001, 0.0, 0.0, 0.008, 0.0, 0.008, -0.00698, 0.0)

In [8]:
const f₁₁_mat = 1*1e-3
const f₂₂_mat = 1*1e-3

const F_mat = TensorValue(f₁₁_mat, 0.0, 0.0, f₁₁_mat)

TensorValue{2, 2, Float64, 4}(0.001, 0.0, 0.0, 0.001)

### Micropolar constitutive parameters and operators

Derived from classical elastic constants `E`, `ν` using:
- `λ = Eν/(1−ν²)` (plane stress)  
- `μ = G(1 − 2N²)/(1 − N²)`  
- `κ = 2GN²/(1 − N²)`  
- `γ = 4Gl²`

Key kinematic objects:
- `ε_Skw(∇u, θ)` — skew-symmetric part of relative deformation: `½(∇uᵀ − ∇u) − Eθ`
- `Skw(w, v)` — same quantity built from test functions `(w, v)`
- `E_Matrx` — permutation tensor `[0 1; −1 0]` mapping scalar `θ` to 2×2 antisymmetric tensor


### SIMP interpolation — PEMAP-P scheme (unchanged)

In [9]:
const Emin = 1e-9
const eMin = 1e-9
const E0   = 1.0
const e0   = 1.0

function Em_uu(x)
    return Emin + (x ^ penal_uu) * (E0 - Emin)
end

function Em_uφ(x, P)
    return (eMin + (x ^ penal_uφ) * (e0 - eMin)) * (2*P - 1)^penal_P
end

function Em_φφ(x)
    return (eMin + (x ^ penal_φφ) * (e0 - eMin))
end

function Em_flexo(x)
    return eMin + (x ^ penal_flexo) * (e0 - eMin)
end

Em_flexo (generic function with 1 method)

In [10]:
order  = 1
degree = 2 * order

Ω  = Triangulation(model)
dΩ = Measure(Ω, degree)

# ── Displacement space (vector CG1) ──────────────────────────────────────────
reffe_Disp = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
V0_Disp = TestFESpace(model, reffe_Disp;
    conformity      = :H1,
    dirichlet_tags  = ["tag_1","tag_2"],
    dirichlet_masks = [(true, true),(false, true)] )

U0_Disp = TrialFESpace(V0_Disp, [VectorValue(0.0, 0.0),VectorValue(0.0, 0.0)])

reffe_ElecPot = ReferenceFE(lagrangian,Float64,order)
V0_ElecPot  = TestFESpace(model,reffe_ElecPot;conformity=:H1,
    dirichlet_tags  = ["tag_7","tag_1","tag_3"],
    dirichlet_masks = [true,true,true])
U0_ElecPot = TrialFESpace(V0_ElecPot, [0.0,0.0,0.0])
U0_ElecPotAdj = TrialFESpace(V0_ElecPot, [0.0,0.0,0.0])
ϕh = zero(V0_ElecPot)
# V0_ElecPot = ConstantFESpace(model; field_type=Float64)
# U0_ElecPot = V0_ElecPot

reffe_theta = ReferenceFE(lagrangian, Float64, order)
V0_theta = TestFESpace(model, reffe_theta;
    conformity      = :H1,
    dirichlet_tags  = ["tag_1","tag_2"],
    dirichlet_masks = [true,true])
U0_theta = TrialFESpace(V0_theta, [0.0,0.0])


# ── Multifield spaces [u; θ] ──────────────────────────────────────────────────
V0 = MultiFieldFESpace([V0_Disp, V0_ElecPot, V0_theta])
U0 = MultiFieldFESpace([U0_Disp, U0_ElecPot, U0_theta])


labels = get_face_labeling(model)
LoadTagId = get_tag_from_name(labels ,"LoadBoundary")
Γ_Load = BoundaryTriangulation(model ;tags = LoadTagId)
dΓ_Load = Measure(Γ_Load ,degree)
n_Γ_Load = get_normal_vector(Γ_Load) 

Γ_top = BoundaryTriangulation(model; tags = ["tag_2","tag_8","tag_4"])
dΓ_top = Measure(Γ_top, degree)
n_top = get_normal_vector(Γ_top)

# ── Design variable spaces (DG0 — one DOF per element) ───────────────────────
p_reffe = ReferenceFE(lagrangian, Float64, 0)
Q  = TestFESpace(Ω, p_reffe; vector_type = Vector{Float64})
P  = Q
np = num_free_dofs(P)

# ── Filtered density space (CG1 — continuous, for Helmholtz filter) ───────────
pf_reffe = ReferenceFE(lagrangian, Float64, 1)
Qf = TestFESpace(Ω, pf_reffe; vector_type = Vector{Float64})
Pf = Qf


# ── Bundle ────────────────────────────────────────────────────────────────────
fem_params = (; V0_Disp, U0_Disp, V0_ElecPot, U0_ElecPot, V0_theta, U0_theta, V0, U0, Q, P, Qf, Pf, np, Ω, dΩ, n_Γ_Load, dΓ_Load,n_top,dΓ_top)

println("Number of design elements: np = $np")
println("Displacement DOFs : $(num_free_dofs(U0_Disp))")
println("Electrical DOFs   : $(num_free_dofs(U0_ElecPot))")
println("Rotation DOFs     : $(num_free_dofs(U0_theta))")
println("Total DOFs        : $(num_free_dofs(U0))")

Number of design elements: np = 16875
Displacement DOFs : 34349
Electrical DOFs   : 17100
Rotation DOFs     : 17174
Total DOFs        : 68623


In [11]:
function stepDisp2(fem_params, ph, polh)
    (; U0, V0, dΩ) = fem_params
    γ_ep = 1e10   # penalty parameter
    a((u,ϕ,θ),(w,ψ,v))  = ∫(h *( ((p -> Em_uu(p)) ∘ ph) * (ε(w) ⊙ (σ_Bmod∘(ε(u))) )  
                        + (((p,P) -> Em_uφ(p,P)) ∘ (ph, polh)) * (∇(w) ⊙ (σ_piezo ∘(∇(ϕ))))  
                        - ((p -> Em_φφ(p)) ∘ ph) * (∇(ψ)⋅(D_piezo∘ (∇(ϕ))))  
                        + (((p,P) -> Em_uφ(p,P)) ∘ (ph, polh)) * (∇(ψ)⋅(D_elas∘(ε(u)))) 
                        - ((p -> Em_flexo(p)) ∘ ph) * ((∇(v))⋅ (M_Flexo∘(∇(ϕ))))
                        + ((p -> Em_flexo(p)) ∘ ph) * (∇(ψ)⋅(D_Flexo∘(∇(θ))))
                        + ((p -> Em_uu(p)) ∘ ph) * ((Skw(w,v)) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ))) )  
                        + ((p -> Em_uu(p)) ∘ ph) * (∇(v))⋅ (M_mod∘(∇(θ))) 
                        - ((p -> Em_uu(p)) ∘ ph) * (v*((E_Matrx) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ))))) ))*dΩ + ∫( γ_ep * ((∇(ϕ) - (∇(ϕ)⋅n_top)*n_top) ⋅ (∇(ψ) - (∇(ψ)⋅n_top)*n_top)))*dΓ_top
    
    b((w,ψ,v)) = ∫(w ⋅ forVec)*dΓ_Load

    op  = AffineFEOperator(a, b, U0, V0)
    uh = solve(op)
    return uh, get_matrix(op)
end

stepDisp2 (generic function with 1 method)

In [12]:
elemsize = maximum(get_el_Δ(model))
r  = (6 * elemsize) / (2 * sqrt(3))#(6 * elemsize) / (2 * sqrt(3))   # filter radius
β  = 2                                 # threshold sharpness
η  = 0.5                               # threshold centre

a_f(r, u, v) = r^2 * (∇(v) ⋅ ∇(u))

function Filter(p0; r, fem_params)
    ph = FEFunction(fem_params.P, p0)
    op = AffineFEOperator(fem_params.Pf, fem_params.Qf) do u, v
        ∫(a_f(r, u, v))fem_params.dΩ + ∫(v * u)fem_params.dΩ,
        ∫(v * ph)fem_params.dΩ
    end
    pfh = solve(op)
    return get_free_dof_values(pfh)
end

function Threshold(pfh; β, η)
    return (tanh(β*η) + tanh(β*(pfh - η))) / (tanh(β*η) + tanh(β*(1.0 - η)))
end

Dptdpf(pf; β, η) = β*(1.0 - tanh(β*(pf - η))^2) / (tanh(β*η) + tanh(β*(1.0 - η)))

Dptdpf (generic function with 1 method)

### Sensitivity derivatives

In [13]:

DEdpf_uu(pf; β, η) = penal_uu * Threshold(pf; β, η)^(penal_uu-1) * Dptdpf(pf; β, η) * (E0 - Emin)
DEdpf_uφ(pf; β, η) = penal_uφ * Threshold(pf; β, η)^(penal_uφ-1) * Dptdpf(pf; β, η) * (e0 - eMin)
DEdpf_φφ(pf; β, η) = penal_φφ * Threshold(pf; β, η)^(penal_φφ-1) * Dptdpf(pf; β, η) * (e0 - eMin)
DEdpf_flexo(pf; β, η) = penal_flexo * Threshold(pf; β, η)^(penal_flexo-1) * Dptdpf(pf; β, η) * (e0 - eMin)
DEdpol(P)          = penal_P  * (2*P - 1)^(penal_P-1) * 2 * (e0 - eMin)

                        
                        
function DAdpf(u, ϕ, θ, w, ψ, v, pfh, polh; β, η)
    return ( h * ( ((p -> DEdpf_uu(p; β, η)) ∘ pfh) * (ε(w) ⊙ (σ_Bmod∘(ε(u))) ) 
           + ((p -> DEdpf_uφ(p; β, η)) ∘ pfh) * ((P -> (2*P - 1)^penal_P) ∘ polh) * (∇(w) ⊙ (σ_piezo ∘(∇(ϕ))))
           - ((p -> DEdpf_φφ(p; β, η)) ∘ pfh) * (∇(ψ)⋅(D_piezo∘ (∇(ϕ)))) 
           + ((p -> DEdpf_uφ(p; β, η)) ∘ pfh) * ((P -> (2*P - 1)^penal_P) ∘ polh) * (∇(ψ)⋅(D_elas∘(ε(u))))
           - ((p -> DEdpf_flexo(p; β, η)) ∘ pfh) * ((∇(v))⋅ (M_Flexo∘(∇(ϕ))))
           + ((p -> DEdpf_flexo(p; β, η)) ∘ pfh) * (∇(ψ)⋅(D_Flexo∘(∇(θ))))
           + ((p -> DEdpf_uu(p; β, η)) ∘ pfh) * ((Skw(w,v)) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ))) ) 
           + ((p -> DEdpf_uu(p; β, η)) ∘ pfh) * ((∇(v))⋅ (M_mod∘(∇(θ)))) 
           - ((p -> DEdpf_uu(p; β, η)) ∘ pfh) * (v*((E_Matrx) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ))))) ))
end

function DAdpol(u, ϕ, θ, w, ψ, v, pth, polh; β, η)
    return ( h * ( ((P -> DEdpol(P)) ∘ polh) * ((p -> p^penal_uφ) ∘ pth) * (∇(w) ⊙ (σ_piezo ∘(∇(ϕ)))) 
            + ((P -> DEdpol(P)) ∘ polh) * ((p -> p^penal_uφ) ∘ pth) * (∇(ψ)⋅(D_elas∘(ε(u)))) )) 
end

function Df1dpf(u,ϕ,θ,pfh, polh; β, η)
 return ( h * (((p -> DEdpf_uu(p; β, η)) ∘ pfh) * 0.5 * (ε(u) ⊙ (σ_Bmod∘(ε(u))) )))
end

function Dfdpol(u,ϕ,θ,pfh, polh; β, η)
 return 0.0
end

function Df2dpf(u,ϕ,θ,pfh, polh; β, η)            
 return ( h * (((p -> DEdpf_φφ(p; β, η)) ∘ pfh) * 0.5 * (∇(ϕ)⋅(D_piezo∘ (∇(ϕ))))))
end

Df2dpf (generic function with 1 method)

In [14]:
function Dgdp(dgdpf; r, fem_params)
    Af = assemble_matrix(fem_params.Pf, fem_params.Qf) do u, v
        ∫(a_f(r, u, v))fem_params.dΩ + ∫(v * u)fem_params.dΩ
    end
    wvec = Af' \ dgdpf
    wh   = FEFunction(fem_params.Pf, wvec)
    l_temp(dp) = ∫(wh * dp)fem_params.dΩ
    return assemble_vector(l_temp, fem_params.P)
end

Dgdp (generic function with 1 method)

### Sanity check — full solid, uniform polarisation

In [15]:
p0_init   = ones(fem_params.np)
pol0_init = fill(1.0, fem_params.np)

pf_vec_chk = Filter(p0_init; r, fem_params)
pfh_chk    = FEFunction(fem_params.Pf, pf_vec_chk)
pth_chk    = (pf -> Threshold(pf; β, η)) ∘ pfh_chk

polh_chk     = FEFunction(fem_params.P, pol0_init)
polh_eff_chk = Operation(P -> (2*P - 1)^penal_P)(polh_chk)

uh_comb_chk,_ = stepDisp2(fem_params, pth_chk, polh_chk)
uh_u_chk, uh_ϕ_chk, uh_θ_chk = uh_comb_chk

filename1 = "SanityCheck_FullSolid_w_obj$(replace(string(w_obj), "." => "_"))"
writevtk(Ω, filename1, cellfields = [
    "uh"      => uh_u_chk,
    "phi"     => uh_ϕ_chk,
    "theta"   => uh_θ_chk,
    "pth"     => pth_chk,
    "pol"     => polh_chk,
    "pol_eff" => polh_eff_chk
])
println("Sanity-check VTK written: $(filename1).vtu")

Sanity-check VTK written: SanityCheck_FullSolid_w_obj0_5.vtu


In [16]:
J1=sum(∫(((p->Em_uu(p))∘pth_chk)*0.5*h*(ε(uh_u_chk)⊙(σ_Bmod∘ε(uh_u_chk))))*fem_params.dΩ)

J2=sum(∫(((p->Em_φφ(p))∘pth_chk)*0.5*h*(∇(uh_ϕ_chk)⋅(D_piezo∘ (∇(uh_ϕ_chk)))))*fem_params.dΩ)

0.006281772486480036

In [17]:
J1_ref,J2_ref = J1,J2 #1e-20,1e1

(0.11050557084016553, 0.006281772486480036)

### Objective + gradient — `gf_p`

In [18]:
const _gfp_first_call = Ref(true)
function gf_p(vars::Vector, grad::Vector; r, β, η, fem_params)
    np  = fem_params.np
    p0  = vars[1:np]
    pol = vars[np+1:end]

    pf_vec = Filter(p0; r, fem_params)
    pfh    = FEFunction(fem_params.Pf, pf_vec)
    pth    = (pf -> Threshold(pf; β, η)) ∘ pfh

    polh = FEFunction(fem_params.P, pol)

    uh_comb, A_mat = stepDisp2(fem_params, pth, polh)
    uh_u, uh_ϕ, uh_θ = uh_comb

    # J1 = mechanical strain energy
    J1 = sum(∫(((p -> Em_uu(p)) ∘ pth) * 0.5 * h * (ε(uh_u) ⊙ (σ_Bmod∘(ε(uh_u))))) * fem_params.dΩ)
    # J2 = electrical energy
    J2 = sum(∫(((p -> Em_φφ(p)) ∘ pth) * 0.5 * h * (∇(uh_ϕ)⋅(D_piezo∘ (∇(uh_ϕ))))) * fem_params.dΩ)
    
    J_tot = w_obj * J1 - (1 - w_obj) * J2

    if length(grad) > 0

        nu = num_free_dofs(fem_params.U0_Disp)
        nϕ = num_free_dofs(fem_params.U0_ElecPot)
        nθ = num_free_dofs(fem_params.U0_theta)

        # ════════════════════════════════════════════════════════════════════
        # ADJOINT FOR J1  (mechanical strain energy)
        # ∂J1/∂u = ∫ Em_uu * h * ε(w):C_B:ε(u) dΩ  (0.5 cancels by symmetry)
        # ∂J1/∂ϕ = 0,   ∂J1/∂θ = 0
        # ════════════════════════════════════════════════════════════════════
        df1duVec = assemble_vector(fem_params.U0_Disp) do w
            ∫(((p -> Em_uu(p)) ∘ pth) * h * (ε(w) ⊙ (σ_Bmod∘(ε(uh_u))))) * fem_params.dΩ
        end
        df1dϕVec = zeros(nϕ)
        df1dθVec = zeros(nθ)

        df1duϕθVec = vcat(-df1duVec, -df1dϕVec, -df1dθVec)
        w1vec = A_mat' \ df1duϕθVec

        w1Uconjh = FEFunction(fem_params.U0_Disp,    w1vec[1:nu])
        w1ϕconjh = FEFunction(fem_params.U0_ElecPot, w1vec[nu+1:nu+nϕ])
        w1θconjh = FEFunction(fem_params.U0_theta,   w1vec[nu+nϕ+1:end])

        # Density sensitivity for J1
        w1timsdgdp(dp) =
            ∫(Df1dpf(uh_u, uh_ϕ, uh_θ, pfh, polh; β, η) * dp) * fem_params.dΩ +
            ∫(DAdpf(uh_u, uh_ϕ, uh_θ, w1Uconjh, w1ϕconjh, w1θconjh,
                    pfh, polh; β, η) * dp) * fem_params.dΩ
        dgf1dpf      = assemble_vector(w1timsdgdp, fem_params.Pf)
        grad_density1 = Dgdp(dgf1dpf; r, fem_params)

        # Polarisation sensitivity for J1
        w1timsdgdp_pol(dpol) =
            ∫(DAdpol(uh_u, uh_ϕ, uh_θ, w1Uconjh, w1ϕconjh, w1θconjh,
                     pth, polh; β, η) * dpol) * fem_params.dΩ
        grad_pol1 = assemble_vector(w1timsdgdp_pol, fem_params.P)

        df2duVec = zeros(nu)
        df2dϕVec = assemble_vector(fem_params.U0_ElecPot) do ψ
            ∫(((p -> Em_φφ(p)) ∘ pth) * h * (∇(ψ)⋅(D_piezo∘ (∇(uh_ϕ))))) * fem_params.dΩ
        end
        df2dθVec = zeros(nθ)

        df2duϕθVec = vcat(-df2duVec, -df2dϕVec, -df2dθVec)
        w2vec = A_mat' \ df2duϕθVec

        w2Uconjh = FEFunction(fem_params.U0_Disp,    w2vec[1:nu])
        w2ϕconjh = FEFunction(fem_params.U0_ElecPot, w2vec[nu+1:nu+nϕ])
        w2θconjh = FEFunction(fem_params.U0_theta,   w2vec[nu+nϕ+1:end])

        # Density sensitivity for J2
        w2timsdgdp(dp) =
            ∫(Df2dpf(uh_u, uh_ϕ, uh_θ, pfh, polh; β, η) * dp) * fem_params.dΩ +
            ∫(DAdpf(uh_u, uh_ϕ, uh_θ, w2Uconjh, w2ϕconjh, w2θconjh,
                    pfh, polh; β, η) * dp) * fem_params.dΩ
        dgf2dpf       = assemble_vector(w2timsdgdp, fem_params.Pf)
        grad_density2 = Dgdp(dgf2dpf; r, fem_params)

        w2timsdgdp_pol(dpol) =
            ∫(DAdpol(uh_u, uh_ϕ, uh_θ, w2Uconjh, w2ϕconjh, w2θconjh,
                     pth, polh; β, η) * dpol) * fem_params.dΩ
        grad_pol2 = assemble_vector(w2timsdgdp_pol, fem_params.P)

        grad_density = w_obj .* grad_density1 .- (1 - w_obj) .* grad_density2
        grad_pol     = w_obj .* grad_pol1     .- (1 - w_obj) .* grad_pol2


        grad[:] = vcat(collect(grad_density), collect(grad_pol))
    end

    polh_eff = Operation(P -> (2*P - 1)^penal_P)(polh)
    fname    = "OptResults_w_obj$(replace(string(w_obj), "." => "_"))"
    writevtk(fem_params.Ω, fname, cellfields = [
        "uh"       => uh_u,
        "phi"      => uh_ϕ,
        "theta"    => uh_θ,
        "pth"      => pth,
        "polh"     => polh,
        "polh_eff" => polh_eff,
        "σ_Bmod"   => (σ_Bmod∘(ε(uh_u))),
        "σ_piezo"  => (σ_piezo ∘(∇(uh_ϕ))),
        "D_elas"  => (D_elas∘(ε(uh_u)))
    ])
    grad_density_h = FEFunction(fem_params.P,  collect(grad_density))
    grad_pol_h     = FEFunction(fem_params.P,  collect(grad_pol))
    writevtk(fem_params.Ω, "grad_debug", cellfields = [
            "grad_density" => grad_density_h,
            "grad_pol"     => grad_pol_h
        ])
     mode = _gfp_first_call[] ? "w" : "a"
    open("gvaluemma.txt", mode) do io
        write(io, "$J_tot\n")
    end
    open("J1_values.txt", mode) do io
        write(io, "$J1\n")
    end
    open("J2_values.txt", mode) do io
        write(io, "$J2\n")
    end
    _gfp_first_call[] = false

    return J_tot
end

gf_p (generic function with 1 method)

### Volume constraint — unchanged (depends only on density, not microrotation)

In [19]:
const _cfp_first_call = Ref(true)
dv = get_array(∫(1)fem_params.dΩ)
function cf_p(vars::Vector, gradc::Vector; r, β, η, fem_params)
    np  = fem_params.np
    p0  = vars[1:np]
    pf_vec = Filter(p0; r, fem_params)
    pfh    = FEFunction(fem_params.Pf, pf_vec)
    if length(gradc) > 0
        # Volume gradient: sensitive to density only, zero for polarisation
        gradc[:] = vcat(dv, zeros(np))
    end

    cval = sum(∫(pfh) * fem_params.dΩ) - volfrac * sum(dv)

    mode = _cfp_first_call[] ? "w" : "a"
    open("cvalue.txt", mode) do io
        write(io, "$cval\n")
    end
    _cfp_first_call[] = false

    return cval
end

cf_p (generic function with 1 method)

### Finite-difference gradient check

In [20]:
np_chk = fem_params.np

p0_fd   = fill(volfrac, np_chk)
pol0_fd = fill(0.1, np_chk)
vars0   = vcat(p0_fd, pol0_fd)

δ = 1e-3

# Density perturbation at element 30
δp_disp        = zeros(2 * np_chk)
δp_disp[600]    = δ

# Polarisation perturbation at element 30
δp_pol         = zeros(2 * np_chk)
δp_pol[np_chk + 600] = δ

grad  = zeros(2 * np_chk)
gradm = zeros(2 * np_chk)

g0      = gf_p(vars0,            grad;  r, β, η, fem_params)
g1_disp = gf_p(vars0 + δp_disp, gradm; r, β, η, fem_params)
g1_pol  = gf_p(vars0 + δp_pol,  gradm; r, β, η, fem_params)

println("==========================================")
println("FD CHECK — DENSITY")
println("==========================================")
println("FD approximation  : $(g1_disp - g0)")
println("Adjoint prediction: $(grad' * δp_disp)")
println("Relative error    : $((g1_disp - g0 - grad'*δp_disp) / (g1_disp - g0))")
println("==========================================")
println("FD CHECK — POLARISATION")
println("==========================================")
println("FD approximation  : $(g1_pol - g0)")
println("Adjoint prediction: $(grad' * δp_pol)")
println("Relative error    : $((g1_pol - g0 - grad'*δp_pol) / (g1_pol - g0))")
println("==========================================")

FD CHECK — DENSITY
FD approximation  : -1.3867919843590926e-6
Adjoint prediction: -1.3868968798094488e-6
Relative error    : -7.563892172678997e-5
FD CHECK — POLARISATION
FD approximation  : -1.6670886893166426e-10
Adjoint prediction: -1.665979853417569e-10
Relative error    : 0.0006651331186993305


### MMA optimiser — `gf_p_optimize`

In [21]:
function gf_p_optimize(vars_init::Vector; r, β, η, TOL=1e-4, MAX_ITER, fem_params)
    np = fem_params.np

    opt = Opt(:LD_MMA, 2 * np)

    # Bounds: density ∈ [0.001, 1],  polarisation ∈ [0, 1]
    opt.lower_bounds = vcat(fill(0.001, np), fill(0.0, np))
    opt.upper_bounds = vcat(fill(1.0,   np), fill(1.0, np))

    opt.xtol_rel  = TOL
    opt.maxeval   = MAX_ITER

    opt.min_objective = (vars, grad) -> gf_p(vars, grad; r, β, η, fem_params)

    inequality_constraint!(opt,
        (vars, gradc) -> cf_p(vars, gradc; r, β, η, fem_params), 1e-8)

    (g_opt, vars_opt, ret) = optimize(opt, vars_init)

    p_opt   = vars_opt[1:np]
    pol_opt = vars_opt[np+1:end]

    @show numevals = opt.numevals
    println("got $(g_opt) after $numevals iterations (returned $ret)")

    return g_opt, vars_opt, ret
end

gf_p_optimize (generic function with 1 method)

### Run the optimisation

In [22]:
np_run = fem_params.np

p0_run   = fill(volfrac, np_run)
pol0_run = fill(1.0, np_run)

vars0_run = vcat(p0_run, pol0_run)

TOL      = 1e-12
MAX_ITER = 1000

g_opt, vars_opt, ret = gf_p_optimize(vars0_run; r, β, η, TOL, MAX_ITER, fem_params)

p_opt   = vars_opt[1:np_run]
pol_opt = vars_opt[np_run+1:end]

@printf "Final density  ∈ [%.4f, %.4f]\n" minimum(p_opt)   maximum(p_opt)
@printf "Final pol      ∈ [%.4f, %.4f]\n" minimum(pol_opt) maximum(pol_opt)

numevals = opt.numevals = 384
got 0.09539158091377606 after 384 iterations (returned XTOL_REACHED)
Final density  ∈ [0.0010, 1.0000]
Final pol      ∈ [0.0000, 1.0000]


### Post-processing — final VTK with sharp threshold

In [23]:
β_post = 64    # sharper threshold for crisp boundary visualisation

pf_vec_opt = Filter(p_opt; r, fem_params)
pfh_opt    = FEFunction(fem_params.Pf, pf_vec_opt)
pth_opt    = (pf -> Threshold(pf; β=β_post, η)) ∘ pfh_opt

polh_opt     = FEFunction(fem_params.P, pol_opt)
polh_eff_opt = Operation(P -> (2*P - 1)^penal_P)(polh_opt)

uh_comb_opt, _ = stepDisp2(fem_params, pth_opt, polh_opt)
uh_u_opt, uh_ϕ_opt, uh_θ_opt = uh_comb_opt

fname_final = "FinalResult_w_obj$(replace(string(w_obj), "." => "_"))"
writevtk(Ω, fname_final, cellfields = [
    "uh"       => uh_u_opt,
    "phi"      => uh_ϕ_opt,
    "theta"    => uh_θ_opt,
    "pth"      => pth_opt,
    "pol_opt"  => polh_opt,
    "pol_eff"  => polh_eff_opt
])
println("Final VTK written: $(fname_final).vtu")

Final VTK written: FinalResult_w_obj0_5.vtu
